In [ ]:
# Core libraries
import pandas as pd
import numpy as np
from pathlib import Path

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

---

## 0. Data Scraping

Scape the data from https://store.steampowered.com/appreviews/ and store them in a csv file.

In [175]:
import requests
import pandas as pd
import time
from pathlib import Path
from typing import List, Dict, Optional
import logging

In [ ]:
def scrape_steam_reviews(app_id: int, max_reviews: int = 50000, language: str = 'english', reviews_per_page: int = 100) -> List[Dict]:
    all_reviews = []
    cursor = '*'
    
    while len(all_reviews) < max_reviews:
        url = f'https://store.steampowered.com/appreviews/{app_id}'
        params = {
            'json': 1,
            'language': language,
            'cursor': cursor,
            'num_per_page': reviews_per_page,
            'filter': 'recent'
        }
        
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            data = response.json()
            
            if not data.get('reviews'):
                break
            
            all_reviews.extend(data['reviews'])
            cursor = data.get('cursor')
            
            if not cursor:
                break
            
            time.sleep(1)
            
        except requests.exceptions.RequestException as e:
            time.sleep(10)
            continue
    
    return all_reviews[:max_reviews]

In [ ]:
def fetch_game_details(app_id: int) -> Dict:
    url = f'https://store.steampowered.com/api/appdetails'
    params = {'appids': app_id}
    
    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if str(app_id) in data and data[str(app_id)]['success']:
            return data[str(app_id)]['data']
        else:
            return {}
            
    except requests.exceptions.RequestException:
        return {}


def process_reviews(reviews: List[Dict], game_data: Dict = None) -> pd.DataFrame:
    """Process reviews to extract features from Steam API."""
    data = []
    
    for review in reviews:
        author = review.get('author', {})
        
        record = {
            # Review features
            'recommended': review.get('voted_up', False),
            'review_text': review.get('review', ''),
            'review_length': len(review.get('review', '')),
            'playtime_at_review_hours': author.get('playtime_at_review', 0) / 60,
            'playtime_total_hours': author.get('playtime_forever', 0) / 60,
            'received_free': review.get('received_for_free', False),
            'written_early_access': review.get('written_during_early_access', False),
            
            # User features
            'user_id': author.get('steamid', ''),
            'user_games_owned': author.get('num_games_owned', 0),
            'user_num_reviews': author.get('num_reviews', 0),
            
            # Review metadata
            'votes_helpful': review.get('votes_up', 0),
            'votes_funny': review.get('votes_funny', 0),
            'weighted_vote_score': review.get('weighted_vote_score', 0.5),
            'comment_count': review.get('comment_count', 0),
            'timestamp_created': review.get('timestamp_created', 0),
            'timestamp_updated': review.get('timestamp_updated', 0),
            'review_id': review.get('recommendationid', 0),
        }
        
        # Game features
        if game_data:
            record.update({
                'game_id': game_data.get('steam_appid', 0),
                'game_name': game_data.get('name', ''),
                'game_price': game_data.get('price_overview', {}).get('final', 0) / 100 if game_data.get('price_overview') else 0,
                'game_required_age': game_data.get('required_age', 0),
                'game_genre': ','.join([g['description'] for g in game_data.get('genres', [])]),
                'game_categories': ','.join([c['description'] for c in game_data.get('categories', [])]),
                'game_developer': ','.join(game_data.get('developers', [])),
                'game_publisher': ','.join(game_data.get('publishers', [])),
                'game_description': game_data.get('short_description', ''),
                'game_is_free': game_data.get('is_free', False),
            })
        
        data.append(record)
    
    return pd.DataFrame(data)

In [ ]:
# Scrape reviews from multiple games
GAMES = [236430, 1517290, 1938090, 1203220, 2357570, 976730, 1551360, 1294810, 954850, 1086940]
REVIEWS_PER_GAME = 2500
OUTPUT_FILE = 'steam_reviews_multi.csv'

all_game_dfs = []

for idx, app_id in enumerate(GAMES):
    try:
        game_data = fetch_game_details(app_id)
        time.sleep(2)
        
        reviews = scrape_steam_reviews(app_id=app_id, max_reviews=REVIEWS_PER_GAME)
        
        if len(reviews) > 0:
            df_game = process_reviews(reviews, game_data=game_data)
            all_game_dfs.append(df_game)
        
        time.sleep(5)
        
    except Exception:
        continue

if all_game_dfs:
    df = pd.concat(all_game_dfs, ignore_index=True)
    df.to_csv(OUTPUT_FILE, index=False)

In [180]:
print(f"Dataset shape: {df.shape}")
print(f"Positive reviews: {df['recommended'].sum()} ({df['recommended'].mean()*100:.1f}%)")
print(f"Number of unique games: {df['game_id'].nunique()}")

print(f"\nGames in dataset:")
for game_id in df['game_id'].unique():
    game_df = df[df['game_id'] == game_id]
    game_name = game_df['game_name'].iloc[0]
    pos_pct = game_df['recommended'].mean() * 100
    print(f"  {game_name}: {len(game_df)} reviews ({pos_pct:.1f}% positive)")

print(f"\nFirst few rows:")
df.head()

Dataset shape: (21753, 27)
Positive reviews: 13966 (64.2%)
Number of unique games: 10

Games in dataset:
  DARK SOULS™ II: 2500 reviews (82.8% positive)
  Battlefield™ 2042: 2500 reviews (47.6% positive)
  Call of Duty®: 2500 reviews (43.8% positive)
  NARAKA: BLADEPOINT: 2500 reviews (80.7% positive)
  Overwatch® 2: 124 reviews (45.2% positive)
  Halo: The Master Chief Collection: 2500 reviews (82.6% positive)
  Forza Horizon 5: 2500 reviews (86.9% positive)
  Redfall: 1629 reviews (41.4% positive)
  Kerbal Space Program 2: 2500 reviews (9.5% positive)
  Baldur's Gate 3: 2500 reviews (95.5% positive)

First few rows:


,recommended,review_text,review_length,playtime_at_review_hours,playtime_total_hours,received_free,written_early_access,user_id,user_games_owned,user_num_reviews,...,game_id,game_name,game_price,game_required_age,game_genre,game_categories,game_developer,game_publisher,game_description,game_is_free
0,False,Actually kind of insane how boring this game i...,452,1.016667,1.016667,False,False,76561199549899273,0,22,...,236430,DARK SOULS™ II,147.0,0,"Action,RPG","Single-player,Multi-player,Co-op,Steam Achieve...","FromSoftware, Inc.","BANDAI NAMCO Entertainment,FromSoftware, Inc.","Developed by FROM SOFTWARE, DARK SOULS™ II is ...",False
1,True,Dark Souls 2: 8/10. It's amazing.\n\nDark Soul...,93,0.233333,0.233333,False,False,76561198053944201,145,9,...,236430,DARK SOULS™ II,147.0,0,"Action,RPG","Single-player,Multi-player,Co-op,Steam Achieve...","FromSoftware, Inc.","BANDAI NAMCO Entertainment,FromSoftware, Inc.","Developed by FROM SOFTWARE, DARK SOULS™ II is ...",False
2,True,Things to do if you buy Dark souls 2 : uninstall,48,6.700000,10.266667,False,False,76561199815144195,0,8,...,236430,DARK SOULS™ II,147.0,0,"Action,RPG","Single-player,Multi-player,Co-op,Steam Achieve...","FromSoftware, Inc.","BANDAI NAMCO Entertainment,FromSoftware, Inc.","Developed by FROM SOFTWARE, DARK SOULS™ II is ...",False
3,False,"Yeah, it's the same game from the SotFS editio...",518,199.566667,199.566667,False,False,76561198004038352,1862,284,...,236430,DARK SOULS™ II,147.0,0,"Action,RPG","Single-player,Multi-player,Co-op,Steam Achieve...","FromSoftware, Inc.","BANDAI NAMCO Entertainment,FromSoftware, Inc.","Developed by FROM SOFTWARE, DARK SOULS™ II is ...",False
4,False,Not worth it,12,87.583333,87.583333,False,False,76561199117526837,222,4,...,236430,DARK SOULS™ II,147.0,0,"Action,RPG","Single-player,Multi-player,Co-op,Steam Achieve...","FromSoftware, Inc.","BANDAI NAMCO Entertainment,FromSoftware, Inc.","Developed by FROM SOFTWARE, DARK SOULS™ II is ...",False


---

## 1. Data Loading & Exploration

Load the preprocessed Steam reviews dataset and perform exploratory analysis.

In [248]:
from pathlib import Path


In [ ]:
DATA_FILE = Path('steam_reviews_multi.csv')

if not DATA_FILE.exists():
    raise FileNotFoundError(f"Data file not found: {DATA_FILE}")

df = pd.read_csv(DATA_FILE)

print(f"Dataset shape: {df.shape}")
df.head()

In [250]:
print(f"Total records: {len(df):,}")
print(f"Positive reviews: {df['recommended'].sum():,} ({df['recommended'].mean()*100:.1f}%)")
print(f"Negative reviews: {(~df['recommended']).sum():,} ({(~df['recommended']).mean()*100:.1f}%)")

Total records: 21,753
Positive reviews: 13,966 (64.2%)
Negative reviews: 7,787 (35.8%)


In [ ]:
# Data quality check
print("Missing values:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)
print("\nBasic statistics:")
df.describe()

---

## 2. Feature Engineering

Prepare features from review text, playtime, and other variables for the binary classifier.

In [252]:
# Feature engineering imports
from textblob import TextBlob


In [ ]:
# Feature engineering
df['review_text'] = df['review_text'].fillna('')
df['review_length'] = df['review_length'].fillna(0)

# User/Review features
df['log_playtime_at_review'] = np.log1p(df['playtime_at_review_hours'])
df['playtime_ratio'] = df['playtime_at_review_hours'] / (df['playtime_total_hours'] + 1)
df['user_experience_level'] = (df['user_games_owned'] > 50).astype(int)
df['review_engagement'] = df['votes_helpful'] + df['votes_funny']

# Entity features - numeric
df['game_price'] = df['game_price'].fillna(0)
df['game_required_age'] = df['game_required_age'].fillna(0)
df['game_description_length'] = df['game_description'].fillna('').apply(lambda x: len(str(x).split()))

# Entity features - categorical
df['game_genre'] = df['game_genre'].fillna('Unknown')
df['has_multiplayer'] = df['game_categories'].fillna('').apply(lambda x: 1 if 'Multi-player' in str(x) else 0)
df['has_coop'] = df['game_categories'].fillna('').apply(lambda x: 1 if 'Co-op' in str(x) else 0)
df['has_achievements'] = df['game_categories'].fillna('').apply(lambda x: 1 if 'Steam Achievements' in str(x) else 0)

# Entity features - text sentiment
from textblob import TextBlob
df['description_sentiment'] = df['game_description'].fillna('').apply(
    lambda x: TextBlob(str(x)).sentiment.polarity if str(x) else 0
)

# Select final feature set
df = df[[
    'recommended',
    'review_text',
    'log_playtime_at_review',
    'playtime_ratio',
    'user_experience_level',
    'review_length',
    'review_engagement',
    'game_price',
    'game_required_age',
    'game_description_length',
    'description_sentiment',
    'game_genre',
    'has_multiplayer',
    'has_coop',
    'has_achievements'
]]

print(f"Dataset shape: {df.shape}")
print(f"\nFeature breakdown:")
print(f"  FREE TEXT: review_text (TF-IDF)")
print(f"  USER/REVIEW: log_playtime_at_review, playtime_ratio, user_experience_level, review_length, review_engagement")
print(f"  ENTITY NUMERIC: game_price, game_required_age, game_description_length, description_sentiment")
print(f"  ENTITY CATEGORICAL: game_genre, has_multiplayer, has_coop, has_achievements")
df.head(10)

---

## 3. Model Training

Train the binary classifier to predict whether a user will recommend a game.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import time
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
def train_classifier(X_train, y_train, model=None):
    """Train binary classifier to predict game recommendations."""
    from sklearn.preprocessing import StandardScaler, OneHotEncoder
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.compose import ColumnTransformer
    from sklearn.pipeline import Pipeline
    from sklearn.linear_model import LogisticRegression
    
    # Feature groups
    text_features = ['review_text']
    numeric = ['log_playtime_at_review', 'playtime_ratio', 'user_experience_level', 
               'review_length', 'review_engagement',
               'game_price', 'game_required_age', 'game_description_length', 'description_sentiment',
               'has_multiplayer', 'has_coop', 'has_achievements']
    categorical = ['game_genre']
    
    # Preprocessing pipeline
    preprocessor = ColumnTransformer([
        ('text', TfidfVectorizer(max_features=100, min_df=5, max_df=0.8, ngram_range=(1,2)), 'review_text'),
        ('num', StandardScaler(), numeric),
        ('cat', OneHotEncoder(handle_unknown='ignore', max_categories=15), categorical)
    ])
    
    if model is None:
        model = LogisticRegression(random_state=42, max_iter=1000)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    pipeline.fit(X_train, y_train)
    
    return pipeline

In [ ]:
X = df.drop('recommended', axis=1)
y = df['recommended']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train: {len(X_train)} samples, Test: {len(X_test)} samples")
print(f"Train class balance: {y_train.mean():.2%} positive")
print(f"Test class balance: {y_test.mean():.2%} positive")

# Train multiple models
pipeline_lr = train_classifier(X_train, y_train, model=LogisticRegression(random_state=42, max_iter=1000))
pipeline_rf = train_classifier(X_train, y_train, model=RandomForestClassifier(n_estimators=100, random_state=42))
pipeline_xgb = train_classifier(X_train, y_train, model=XGBClassifier(random_state=42, eval_metric='logloss'))
pipeline_svm = train_classifier(X_train, y_train, model=SVC(probability=True, random_state=42))

In [ ]:
# Cross-validation
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(pipeline_lr, X, y, cv=cv, scoring='roc_auc')
print(f"Cross-validated AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# Model comparison
models_to_test = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss'),
    'SVM': SVC(probability=True, random_state=42)
}

results = []
for model_name, model in models_to_test.items():
    start_time = time.time()
    pipeline = train_classifier(X_train, y_train, model=model)
    training_time = time.time() - start_time
    
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1]
    
    results.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC-ROC': roc_auc_score(y_test, y_proba),
        'Training Time (s)': training_time
    })
    print(f"{model_name}: Acc={results[-1]['Accuracy']:.4f}, AUC={results[-1]['AUC-ROC']:.4f}, Time={training_time:.2f}s")

results_df = pd.DataFrame(results).sort_values('Accuracy', ascending=False)

# Calculate overall score
norm_accuracy = (results_df['Accuracy'] - results_df['Accuracy'].min()) / (results_df['Accuracy'].max() - results_df['Accuracy'].min())
norm_auc = (results_df['AUC-ROC'] - results_df['AUC-ROC'].min()) / (results_df['AUC-ROC'].max() - results_df['AUC-ROC'].min())
norm_speed = 1 - ((results_df['Training Time (s)'] - results_df['Training Time (s)'].min()) / (results_df['Training Time (s)'].max() - results_df['Training Time (s)'].min()))
results_df['Overall Score'] = (norm_accuracy * 0.4 + norm_auc * 0.4 + norm_speed * 0.2)
results_df

In [ ]:
# Model comparison dashboard
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Model Comparison Dashboard', fontsize=16, fontweight='bold')
colors = sns.color_palette("husl", len(results_df))

# Accuracy comparison
ax1 = axes[0, 0]
bars = ax1.bar(range(len(results_df)), results_df['Accuracy'], color=colors, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
ax1.set_title('Model Accuracy Comparison', fontsize=12, fontweight='bold')
ax1.set_xticks(range(len(results_df)))
ax1.set_xticklabels(results_df['Model'], rotation=45, ha='right', fontsize=9)
ax1.set_ylim([0.75, 0.85])
ax1.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, results_df['Accuracy']):
    ax1.text(bar.get_x() + bar.get_width()/2, val + 0.002, f'{val:.4f}', ha='center', fontsize=9, fontweight='bold')

# Accuracy vs Training Time trade-off
ax2 = axes[0, 1]
for i, (_, row) in enumerate(results_df.iterrows()):
    ax2.scatter(row['Training Time (s)'], row['Accuracy'], s=300, c=[colors[i]], alpha=0.7, edgecolors='black', linewidth=2)
    ax2.annotate(row['Model'], (row['Training Time (s)'], row['Accuracy']), xytext=(5, 5), textcoords='offset points',
                fontsize=9, fontweight='bold', bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8, edgecolor='black'))
ax2.set_xlabel('Training Time (seconds, log scale)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
ax2.set_title('Accuracy vs Training Time Trade-off', fontsize=12, fontweight='bold')
ax2.set_xscale('log')
ax2.grid(True, alpha=0.3)
median_time, median_acc = results_df['Training Time (s)'].median(), results_df['Accuracy'].median()
ax2.axvline(median_time, color='gray', linestyle='--', alpha=0.5, linewidth=1)
ax2.axhline(median_acc, color='gray', linestyle='--', alpha=0.5, linewidth=1)

# Training time comparison
ax3 = axes[1, 0]
results_time_sorted = results_df.sort_values('Training Time (s)', ascending=False)
bars = ax3.bar(range(len(results_time_sorted)), results_time_sorted['Training Time (s)'], color=colors, edgecolor='black', linewidth=1.5)
ax3.set_ylabel('Training Time (seconds, log scale)', fontsize=11, fontweight='bold')
ax3.set_title('Training Time Comparison', fontsize=12, fontweight='bold')
ax3.set_yscale('log')
ax3.set_xticks(range(len(results_time_sorted)))
ax3.set_xticklabels(results_time_sorted['Model'], rotation=45, ha='right', fontsize=9)
ax3.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, results_time_sorted['Training Time (s)']):
    ax3.text(bar.get_x() + bar.get_width()/2, val * 1.15, f'{val:.2f}s', ha='center', fontsize=9, fontweight='bold')

# Normalized performance metrics
ax4 = axes[1, 1]
x = np.arange(len(results_df))
width = 0.25
ax4.bar(x - width, norm_accuracy, width, label='Accuracy (norm)', color='steelblue', alpha=0.8, edgecolor='black')
ax4.bar(x, norm_auc, width, label='AUC-ROC (norm)', color='coral', alpha=0.8, edgecolor='black')
ax4.bar(x + width, norm_speed, width, label='Speed (norm)', color='lightgreen', alpha=0.8, edgecolor='black')
ax4.set_xlabel('Model', fontsize=11, fontweight='bold')
ax4.set_ylabel('Normalized Score (0-1)', fontsize=11, fontweight='bold')
ax4.set_title('Normalized Performance Metrics', fontsize=12, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels(results_df['Model'], rotation=45, ha='right', fontsize=9)
ax4.legend(loc='upper left', fontsize=9)
ax4.set_ylim([0, 1.1])
ax4.grid(axis='y', alpha=0.3)

# Overall score line
ax4_twin = ax4.twinx()
ax4_twin.plot(x, results_df['Overall Score'], 'o-', color='red', linewidth=2.5, markersize=8, label='Overall Score')
ax4_twin.set_ylabel('Overall Score', fontsize=11, fontweight='bold', color='red')
ax4_twin.tick_params(axis='y', labelcolor='red')
ax4_twin.set_ylim([0, 1.1])
ax4_twin.legend(loc='upper right', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\nBest Accuracy: {results_df.iloc[0]['Model']} ({results_df.iloc[0]['Accuracy']:.4f})")
best_overall_idx = results_df['Overall Score'].idxmax()
print(f"Best Overall Score: {results_df.loc[best_overall_idx, 'Model']} ({results_df.loc[best_overall_idx, 'Overall Score']:.4f})")

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, RocCurveDisplay, PrecisionRecallDisplay

# Evaluation with XGBoost (best model)
y_pred = pipeline_xgb.predict(X_test)
y_proba = pipeline_xgb.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Recommended', 'Recommended']))

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay.from_estimator(pipeline_xgb, X_test, y_test, ax=axes[0], cmap='Blues')
axes[0].set_title("Confusion Matrix - XGBoost", fontsize=12, fontweight='bold')
axes[0].grid(False)

RocCurveDisplay.from_estimator(pipeline_xgb, X_test, y_test, ax=axes[1])
axes[1].set_title("ROC Curve - XGBoost", fontsize=12, fontweight='bold')
axes[1].grid(alpha=0.3)

PrecisionRecallDisplay.from_estimator(pipeline_xgb, X_test, y_test, ax=axes[2])
axes[2].set_title("Precision-Recall Curve - XGBoost", fontsize=12, fontweight='bold')
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance analysis
feature_importance = pipeline_xgb.named_steps['classifier'].feature_importances_
feature_names = pipeline_xgb.named_steps['preprocessor'].get_feature_names_out()

# Top 15 features
indices = np.argsort(feature_importance)[-15:][::-1]
top_features = [(feature_names[i], feature_importance[i]) for i in indices]

plt.figure(figsize=(10, 6))
plt.barh(range(len(top_features)), [f for _, f in top_features])
plt.yticks(range(len(top_features)), [n for n, _ in top_features])
plt.xlabel('Feature Importance (XGBoost)')
plt.title('Top 15 Most Important Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nTop 5 Features:")
for i, (name, importance) in enumerate(top_features[:5], 1):
    print(f"  {i}. {name}: {importance:.4f}")

---

## 4. Prediction Function (with Counterfactuals)

Create a function that:
- Takes a user + game input
- Outputs binary prediction (yes/no they'll like it)
- If prediction is negative: generates counterfactual explanations

In [298]:
def predict_with_counterfactuals(model, user_features, threshold=0.5):
    prediction = model.predict(user_features)[0]
    prob = model.predict_proba(user_features)[0][1]

    counterfactuals = []
    if not prediction:
        feature_names = user_features.columns.tolist()
        counterfactuals = generate_counterfactuals(model, user_features, feature_names)

    return bool(prediction), float(prob), counterfactuals


In [281]:
import dice_ml

def generate_counterfactuals_dice(model, user_entity_features, train_data, num_cfs=3):
    """Generate counterfactuals using DiCE library."""
    # Create DiCE Data object
    d = dice_ml.Data(
        dataframe=train_data,
        continuous_features=['log_playtime_at_review', 'playtime_ratio', 'user_experience_level', 
                           'review_length', 'review_engagement',
                           'game_price', 'game_required_age', 'game_description_length', 'description_sentiment',
                           'has_multiplayer', 'has_coop', 'has_achievements'],
        outcome_name='recommended'
    )
    
    # Create DiCE Model wrapper
    m = dice_ml.Model(model=model, backend="sklearn")
    
    # Create DiCE Explainer
    exp = dice_ml.Dice(d, m, method="random")
    
    # Generate counterfactuals
    dice_exp = exp.generate_counterfactuals(
        user_entity_features,
        total_CFs=num_cfs,
        desired_class="opposite",
        features_to_vary=['game_price', 'game_required_age', 'game_description_length', 'description_sentiment',
                         'game_genre', 'has_multiplayer', 'has_coop', 'has_achievements']
    )
    
    return dice_exp

In [282]:
from anchor import anchor_tabular
import numpy as np

def generate_anchors_explanation(model, user_features, train_data, threshold=0.9):
    """Generate anchor explanations."""
    feature_cols = ['log_playtime_at_review', 'word_count', 'game_price', 'game_is_free', 
                   'game_required_age', 'game_description_length', 'has_multiplayer']
    
    train_array = train_data[feature_cols].values
    
    explainer = anchor_tabular.AnchorTabularExplainer(
        class_names=['Not Recommended', 'Recommended'],
        feature_names=feature_cols,
        train_data=train_array,
        categorical_names={}
    )
    
    def predict_fn(X):
        df_pred = pd.DataFrame(X, columns=feature_cols)
        return model.predict(df_pred)
    
    user_array = user_features[feature_cols].values[0]
    exp = explainer.explain_instance(user_array, predict_fn, threshold=threshold)
    
    return exp

In [273]:
# Create a list of test cases
test_cases = [
  pd.DataFrame({
      'review_text': ["Amazing game! Great graphics, smooth gameplay, tons of fun with friends. Best purchase ever! Highly recommend!"],
      'log_playtime_at_review': [5.5],
      'playtime_ratio': [0.85],
      'user_experience_level': [1],
      'review_length': [250],
      'review_engagement': [150],
      'game_price': [49.99],
      'game_required_age': [12],
      'game_description_length': [45],
      'description_sentiment': [0.4],
      'game_genre': ['Action'],
      'has_multiplayer': [1],
      'has_coop': [1],
      'has_achievements': [1],
  }),
  pd.DataFrame({
      'review_text': ["Terrible game. Waste of money. Buggy, crashes constantly, boring. Refunded."],
      'log_playtime_at_review': [1.0],
      'playtime_ratio': [0.95],
      'user_experience_level': [1],
      'review_length': [80],
      'review_engagement': [5],
      'game_price': [69.99],
      'game_required_age': [18],
      'game_description_length': [15],
      'description_sentiment': [0.05],
      'game_genre': ['Action'],
      'has_multiplayer': [0],
      'has_coop': [0],
      'has_achievements': [1],
  }),
  pd.DataFrame({
      'review_text': ["Absolutely amazing! Love spending $70 on a game that crashes every 10 minutes. Best purchase ever! The developers really care about their fans."],
      'log_playtime_at_review': [0.5],  # Less than 1 hour - barely played
      'playtime_ratio': [1.0],
      'user_experience_level': [1],
      'review_length': [145],
      'review_engagement': [89],  # Moderate engagement
      'game_price': [69.99],  # Premium price
      'game_required_age': [17],
      'game_description_length': [120],  # Detailed description
      'description_sentiment': [0.6],  # Optimistic game description
      'game_genre': ['Action'],
      'has_multiplayer': [1],
      'has_coop': [1],
      'has_achievements': [1],
  }),
  pd.DataFrame({
        'review_text': ["This game ruined my life. I can't stop playing it even though I hate it. 2000 hours wasted. Don't make my mistake. Uninstall while you still can."],
        'log_playtime_at_review': [np.log1p(2000)],  # 2000+ hours
        'playtime_ratio': [1.0],
        'user_experience_level': [1],
        'review_length': [145],
        'review_engagement': [500],  # High engagement - controversial reviews get votes
        'game_price': [0.0],  # Free to play (often associated with addictive mechanics)
        'game_required_age': [13],
        'game_description_length': [55],
        'description_sentiment': [0.3],
        'game_genre': ['Action'],
        'has_multiplayer': [1],
        'has_coop': [0],
        'has_achievements': [1],
    })
]

In [302]:
for case in test_cases:
    pred, prob, counterfactuals = predict_with_counterfactuals(pipeline_xgb, case)

    print(f"Review: \"{case['review_text'].values[0][:80]}...\"")
    print(f"Playtime: {np.expm1(case['log_playtime_at_review'].values[0]):.1f} hours")
    print(f"Price: ${case['game_price'].values[0]}")
    print(f"\nPrediction: {'RECOMMENDED' if pred else 'NOT RECOMMENDED'} (confidence: {prob if pred else (1-prob):.1%})\n")

    if not pred and counterfactuals:
        print("Counterfactual suggestions to change recommendation to RECOMMENDED:")
        print(counterfactuals.visualize_as_dataframe(show_only_changes=True).data_df)

    print("-" * 80)
    print("\n")

Review: "Amazing game! Great graphics, smooth gameplay, tons of fun with friends. Best pu..."
Playtime: 243.7 hours
Price: $49.99

Prediction: RECOMMENDED (confidence: 99.6%)

--------------------------------------------------------------------------------


Review: "Terrible game. Waste of money. Buggy, crashes constantly, boring. Refunded...."
Playtime: 1.7 hours
Price: $69.99

Prediction: NOT RECOMMENDED (confidence: 97.8%)

--------------------------------------------------------------------------------


Review: "Absolutely amazing! Love spending $70 on a game that crashes every 10 minutes. B..."
Playtime: 0.6 hours
Price: $69.99

Prediction: NOT RECOMMENDED (confidence: 81.9%)

--------------------------------------------------------------------------------


Review: "This game ruined my life. I can't stop playing it even though I hate it. 2000 ho..."
Playtime: 2000.0 hours
Price: $0.0

Prediction: NOT RECOMMENDED (confidence: 85.8%)

---------------------------------------------